In [1]:

import arlpy.uwapm as pm
import arlpy.plot as plt
import numpy as np
import math


pm.models()

env = pm.create_env2d()
pm.print_env(env)

surface = np.array([[r, 0.5+0.5*np.sin(2*np.pi*0.005*r)] for r in np.linspace(0,1000,1001)])
env['surface'] = surface
#env['bottom_roughness'] = 100
pm.plot_env(env, width=900)


rays = pm.compute_eigenrays(env)
pm.plot_rays(rays, env=env, width=900)

arrivals = pm.compute_arrivals(env)
pm.plot_arrivals(arrivals, width=900)

# convert to a impulse response time series
sampling_rate = 96000
ir = pm.arrivals_to_impulse_response(arrivals, fs=sampling_rate)
plt.plot(np.abs(ir), fs=96000, width=900)

# freq_response = np.fft.fft(ir)
# plt.plot(freq_response, fs = 96000)


                name : arlpy
   bottom_absorption : 0.1
      bottom_density : 1600
    bottom_roughness : 0
   bottom_soundspeed : 1600
               depth : 25
        depth_interp : linear
           frequency : 25000
           max_angle : 80
           min_angle : -80
              nbeams : 0
            rx_depth : 10
            rx_range : 1000
          soundspeed : 1500
   soundspeed_interp : spline
             surface : None
      surface_interp : linear
            tx_depth : 5
   tx_directionality : None
                type : 2D


In [2]:

sample_time = 1/sampling_rate

time = 0
with open("file.txt", "w", encoding="utf-8") as f:
    for i in ir:
        f.write(f"{i} , {time} \n")
        time = time + sample_time


In [3]:
# Create a 1000Hz sin wave that lasts 1 second

# Generate the time vector

start_time = 0 
end_time = 2

time = np.arange(start_time, end_time, sample_time)

frequency = 1000

phase  =  0

amplitude = 1


signal = amplitude * np.sin( 2 * np.pi * frequency * time  + phase)

points_to_plot = 300

plt.plot(time[0:points_to_plot],signal[0:points_to_plot])

print(len(time))


192000


In [4]:
# Convolve the signal with the impulse response

signal_after_channel = np.abs(np.convolve(ir, signal))

points_to_plot = 500

plt.plot(time[0:points_to_plot],signal_after_channel[0:points_to_plot])

In [5]:
import argparse
import logging
import numpy as np
from scipy.io import wavfile
import librosa
import librosa.display
import matplotlib.pyplot as plot
from scipy.signal import butter, lfilter, sosfiltfilt
import wave
import struct

# from https://github.com/alexleun/FSK-generator/tree/main

# Configure logging
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def butter_bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    if lowcut <= 0 or highcut >= nyq or lowcut >= highcut:
        logging.error("Invalid filter cutoff frequencies. Check lowcut and highcut values.")
        return None
    sos = butter(order, [lowcut, highcut], btype='band', fs=fs, output='sos') #Use SOS for numerical stability
    y = sosfiltfilt(sos, data) # Use sosfiltfilt for zero-phase filtering
    return y

def decode_fsk(file_path, frequency, deviation, bit_duration, window_size=2048, hop_length=512, resample_rate=200000):
    try:
        sr, data = wavfile.read(file_path)
        logging.info(f"Original sample rate: {sr} Hz")
    except FileNotFoundError:
        logging.error(f"File not found at {file_path}")
        return
    except wave.Error as e:
        logging.error(f"Error reading WAV file: {e}")
        return

    #Resample to a lower rate
    data = librosa.resample(data.astype(np.float32), orig_sr=sr, target_sr=resample_rate)
    sr = resample_rate
    logging.info(f"Resampled to: {sr} Hz")


    #Improved clipping check and normalization
    data = np.clip(data, -1, 1) #Clip values to prevent issues

    nyquist_frequency = sr / 2
    if frequency + deviation > nyquist_frequency:
        logging.error(f"Carrier frequency + deviation ({frequency + deviation} Hz) exceeds the Nyquist frequency ({nyquist_frequency} Hz). Reduce the carrier frequency or increase the sample rate.")
        return

    lowcut = frequency - deviation - 200
    highcut = frequency + deviation + 200
    filtered_data = butter_bandpass_filter(data, lowcut, highcut, sr)
    if filtered_data is None:
        return

    stft = librosa.stft(filtered_data, n_fft=window_size, hop_length=hop_length)
    magnitudes = np.abs(stft)
    frequencies = librosa.fft_frequencies(sr=sr, n_fft=window_size)

    # ... (rest of the decode_fsk function remains the same)

# ... (rest of the code remains the same)

    decoded_data = ""
    samples_per_bit = int(sr * bit_duration)
    num_frames = int(np.ceil(len(filtered_data) / samples_per_bit)) #Corrected frame calculation


    for i in range(num_frames):
        start_sample = i * samples_per_bit
        end_sample = min((i + 1) * samples_per_bit, len(filtered_data))
        if start_sample >= len(filtered_data) or start_sample == end_sample:
            continue

        frame_magnitudes = magnitudes[:, int(start_sample / hop_length):int(end_sample / hop_length)]

        if frame_magnitudes.size == 0:
            continue

        avg_magnitudes = np.mean(frame_magnitudes, axis=1)
        peak_index = np.argmax(avg_magnitudes)
        peak_frequency = frequencies[peak_index]
        logging.debug(f"Frame {i}: Peak frequency = {peak_frequency:.2f} Hz") #Added logging

        if peak_frequency > frequency + deviation / 2:
            decoded_data += "1"
        else:
            decoded_data += "0"

    num_bits = len(decoded_data)
    total_duration = num_bits * bit_duration
    num_samples = len(data)

    logging.info(f"Decoded data: {decoded_data}")
    logging.info(f"Number of bits: {num_bits}")
    logging.info(f"Total duration: {total_duration:.4f} seconds")
    logging.info(f"Number of samples: {num_samples}")
    logging.info(f"Sample rate used: {sr} Hz")
    logging.info(f"Frequency used: {frequency} Hz")
    logging.info(f"Deviation used: {deviation} Hz")

    # plot.figure(figsize=(10, 4))
    # librosa.display.specshow(librosa.power_to_db(magnitudes, ref=np.max),
    #                          sr=sr, x_axis='time', y_axis='hz')
    # plot.colorbar(format='%+2.0f dB')
    # plot.title('FSK Spectrogram')
    # plot.tight_layout()
    # plot.show()
    return decoded_data



def save_wav_file(file_path, signal, sample_rate):
    """Saves the generated FSK signal to a WAV file."""
    try:
        signal = np.clip(signal, -1, 1)
        signal = (signal * 32767).astype(np.int16)
        with wave.open(file_path, 'w') as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(sample_rate)
            wf.setnframes(len(signal))
            wf.writeframes(signal.tobytes())
        logging.info(f"FSK signal saved to {file_path}")
    except Exception as e:
        logging.exception(f"An error occurred while saving the WAV file: {e}")

def generate_fsk_signal(bits, frequency, deviation, baud_rate, sample_rate=44100):
    """Generates an FSK signal from a binary string using baud rate."""
    try:
        if not all(bit in '01' for bit in bits):
            logging.error("Invalid input: bits string must contain only '0' and '1'.")
            return None

        bit_duration = 1.0 / baud_rate
        num_bits = len(bits)
        total_samples = int(num_bits * bit_duration * sample_rate)
        signal = np.zeros(total_samples)

        mark_frequency = frequency + deviation
        space_frequency = frequency - deviation

        sample_index = 0
        for bit in bits:
            frequency_to_use = mark_frequency if bit == '1' else space_frequency
            num_samples_per_bit = int(bit_duration * sample_rate)
            t = np.linspace(0, bit_duration, num_samples_per_bit, endpoint=False)
            sine_wave = np.sin(2 * np.pi * frequency_to_use * t)
            signal[sample_index:sample_index + num_samples_per_bit] = sine_wave
            sample_index += num_samples_per_bit

        return signal

    except Exception as e:
        logging.exception(f"An unexpected error occurred: {e}")
        return None


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(description='Decode FSK signal')
#     parser.add_argument('file_path', type=str, help='Path to the WAV file')
#     parser.add_argument('--frequency', type=float, required=True, help='Carrier frequency in Hz')
#     parser.add_argument('--deviation', type=float, required=True, help='Frequency deviation in Hz')
#     parser.add_argument('--baud-rate', type=float, required=True, help='Baud rate')
#     parser.add_argument('--debug', type=int, default=20, choices=[10, 20, 30, 40, 50], help='Debug level (10=DEBUG, 20=INFO, 30=WARNING, 40=ERROR, 50=CRITICAL)')
#     args = parser.parse_args()

#     numeric_level = args.debug
#     logging.basicConfig(level=numeric_level)

#     bit_duration = 1.0 / args.baud_rate
#     decode_fsk(args.file_path, args.frequency, args.deviation, bit_duration)







In [6]:
# generate an fsk signal
import random
import string

length = 1000
random_string = ''.join(random.choices(string.ascii_letters + string.digits, k=length))
#print(random_string)
#txt = "Gfgkljhukbklh2345kjhbaweusnJHGk"
bits = ''.join(format(b, '08b') for b in bytearray(random_string, 'utf-8'))



frequency = 40000 #10 kHz
deviation = 200 # 3000 Hz
baud_rate = 50 # 50bps seems to work best

# frequency = 5000 # kHz
# deviation = 1000 # 1000 Hz
# baud_rate = 64 # 64 bits per second

fsk_sig = generate_fsk_signal(bits = bits, frequency = frequency, deviation = deviation, sample_rate = sampling_rate, baud_rate = baud_rate)

# save to wave file
save_wav_file(file_path = "fsk_wave.wav", signal = fsk_sig, sample_rate = sampling_rate)

#decode the fsk signal

decode_fsk(file_path = "fsk_wave.wav", frequency =  frequency, deviation = deviation, 
           bit_duration = 1/baud_rate, window_size=2048, hop_length=512, resample_rate=200000)



'010001100111100001110101011010100100001001110101011011110101010001100101011000110100001101101000011010110011000101101011011110010110110101101101010010100101011101011000010110010110101001111000010011010111010000110111011100110111010101000010010010100011010001100111011001100011000001110111010101000100110101010111010001100101000001100010010011110110101101010101001101010110111101000110010100110100110101001011010101010011011101001011001100000100101101011001010101100110011101110011010100100101100101010110011101110100111001010100011010000110011101110010001100000101010001010011011010000011001001110010010010000111010101101111010011000100111101111000010101000011001101110110011001100011011101101010011010100100011001110111011001100101010001100100010010010110010101111010010010000110101001100101010011100110110000111001010101010111100001010101010110000011001101110110010001010110101101101101011100110101001001010101001100010110010001010111011011110110011101011001010011100110111101010010010100010110000

In [7]:
def compute_signal_energy(signal):
    energy = 0
    for z in signal:
        energy = energy + z**2
    return np.abs(energy)

In [8]:
import numpy as np

def add_awgn(signal, snr_db): # From Claude
    """
    Add AWGN to a signal at a desired SNR.
    
    Parameters:
        signal  : numpy array (real or complex)
        snr_db  : desired SNR in dB
    
    Returns:
        noisy_signal : signal + noise
        noise        : the noise that was added
    """
    snr_linear = 10 ** (snr_db / 10)
    
    signal_power = np.mean(np.abs(signal) ** 2)
    noise_power = signal_power / snr_linear
    
    if np.iscomplexobj(signal):
        # Complex noise: split power equally between I and Q
        noise = np.sqrt(noise_power / 2) * (
            np.random.randn(*signal.shape) + 1j * np.random.randn(*signal.shape)
        )
    else:
        noise = np.sqrt(noise_power) * np.random.randn(*signal.shape)
    
    return signal + noise, noise

In [9]:
def measure_snr(signal, noise):
    sig_pwr = np.mean(np.abs(signal) ** 2)
    noise_pwr = np.mean(np.abs(noise) ** 2)
    return 10 * np.log10(sig_pwr / noise_pwr)


In [10]:
def computeBER(original_bits, decoded_bits):
    num_wrong_bits = 0
    num_bits = 0
    u=zip(original_bits,decoded_bits)
    for i,j in u:
        num_bits +=1
        if i!=j:
            num_wrong_bits += 1
    return num_wrong_bits/num_bits


In [11]:
'''
Simulation Variables
'''
SNRdB = -20 # (dB) SNR of uw channel

TX_GAIN = 10**1 # 10 dB gain from the transmitting circuitry simulation
RX_GAIN = 10**(4) # Upto 94dB gain on the receiver side


'''
Transmission, no amplification
'''
uw_fsk_sig = np.convolve(ir, fsk_sig) # Put the raw signal through the channel

save_wav_file(file_path = "uw_fsk_wave.wav", signal = uw_fsk_sig, 
              sample_rate = sampling_rate)

# Receive and decode the normal bits
decoded_bits_uw_no_amp = decode_fsk(file_path = "uw_fsk_wave.wav", 
                          frequency =  frequency, deviation = deviation, 
           bit_duration = 1/baud_rate, window_size=2048, hop_length=512, resample_rate=200000)


'''
Transmission, WITH amplification on TX and RX
'''

uw_fsk_sig_amped_tx = np.convolve(ir, fsk_sig*TX_GAIN) # Apply the TX gain then put the signal through the channel

amped_uw_fsk_sig = uw_fsk_sig_amped_tx*RX_GAIN # After the channel, 

save_wav_file(file_path = "amped_uw_fsk_wave.wav", signal = amped_uw_fsk_sig,
              sample_rate = sampling_rate)

amped_decoded_bits = decode_fsk(file_path = "amped_uw_fsk_wave.wav", 
                                frequency =  frequency, deviation = deviation, 
           bit_duration = 1/baud_rate, window_size=2048, hop_length=512, resample_rate=200000)

'''
Doing the same thing but adding noise to the amped signal
'''


uw_fsk_sig_amped_tx_noisy, noise = add_awgn(uw_fsk_sig_amped_tx,SNRdB)
measured_snr = measure_snr(uw_fsk_sig_amped_tx_noisy, noise)
print("Measured SNR: ", measured_snr)

amped_uw_fsk_sig_noisy = uw_fsk_sig_amped_tx_noisy * RX_GAIN
save_wav_file(file_path = "amped_uw_fsk_wave_noisy.wav", signal = uw_fsk_sig_amped_tx_noisy,
              sample_rate = sampling_rate)

amped_decoded_bits_noisy = decode_fsk(file_path = "amped_uw_fsk_wave_noisy.wav", 
                                frequency =  frequency, deviation = deviation, 
           bit_duration = 1/baud_rate, window_size=2048, hop_length=512, resample_rate=200000)


'''
Compute energies and BER
'''
no_amp_energy = compute_signal_energy(uw_fsk_sig)
amped_energy_after_channel = compute_signal_energy(uw_fsk_sig_amped_tx)
amped_energy_after_channel_noisy = compute_signal_energy(uw_fsk_sig_amped_tx_noisy)

print("Original Energy: ",no_amp_energy)
print("Amped Energy: ", amped_energy_after_channel)
print("Amped Noisy Energy: ",amped_energy_after_channel_noisy)

BER = computeBER(bits,decoded_bits_uw_no_amp)
amped_BER = computeBER(bits,amped_decoded_bits)
amped_noisy_BER = computeBER(bits,amped_decoded_bits_noisy)
print("Original BER: ",BER)
print("Amped BER: ", amped_BER)
print("Amped BER Noisy: ", amped_noisy_BER)


/tmp/ipykernel_39342/3520757500.py:118: ComplexWarning: Casting complex values to real discards the imaginary part
  signal = (signal * 32767).astype(np.int16)


Measured SNR:  0.043363807715598376
Original Energy:  7.43524701925493
Amped Energy:  743.5247019226865
Amped Noisy Energy:  733.6177015213905
Original BER:  0.1725
Amped BER:  0.1725
Amped BER Noisy:  0.29275
